# SHREDR AI - Instruction Dataset Construction

This notebook creates the instruction tuning dataset used to fine-tune the language model component of SHREDR AI.
The language model does not perform malware classification.
Its role is to convert structured classifier evidence into a concise, consistent malware-triage report for human reivew.
The dataset is synthetically constructured from the defined input schema and operational rules

In [1]:
from pathlib import Path
import json
import random

import pandas as pd
from sklearn.model_selection import train_test_split

random.seed(42)

In [2]:
PROJECT_ROOT = Path.cwd().parent.parent
LLM_DATA_DIR = PROJECT_ROOT / "llm" / "data"

LLM_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("LLM data directory:", LLM_DATA_DIR)

Project root: C:\Users\jeepe\IT8532\SHREDR-AI
LLM data directory: C:\Users\jeepe\IT8532\SHREDR-AI\llm\data


## Dataset Design
Each example contains:
- instruction
- input
- output

The model is trained to generate a short triage report containing:
1. Risk assessment
2. Evidence interpretation
3. Recommended action

The model must not invent evidence that is not shown in the input.

In [3]:
# helper function

def build_example(
    classification,
    probability,
    threshold,
    file_size_group,
    max_entropy,
    signed,
    confidence_band
):
    instruction = (
        "Generate a concise malware triage report from the supplied analysis evidence."
    )

    input_text = (
        f"Classification: {classification}\n"
        f"Malware probability: {probability:.3f}\n"
        f"Decision threshold: {threshold:.2f}\n"
        f"File size group: {file_size_group}\n"
        f"Maximum entropy: {max_entropy:.2f}\n"
        f"Digitally signed: {'yes' if signed else 'no'}"
    )

    if classification == "malware":
        if confidence_band == "high":
            risk = "High"
            explanation = (
                f"The executable was classified as malicious with a probability "
                f"of {probability*100:.1f}%, which is well above the configured "
                f"{threshold*100:.0f}% decision threshold."
            )

            if max_entropy >= 7.0:
                explanation += " The file also has high entropy, which may indicate packing or obfuscation."

            if not signed:
                explanation += " The executable is not digitally signed."

            recommendation = (
                "Do not execute the file. Escalate it for analyst review."
            )

        else:
            risk = "Review required"
            explanation = (
                f"The executable was classified as malicious with a probability "
                f"of {probability*100:.1f}%, but the score is only slightly above "
                f"the configured {threshold*100:.0f}% threshold."
            )

            recommendation = (
                "Hold the file for human review before any further action."
            )

    else:
        risk = "Low"
        explanation = (
            f"The executable was classified as benign with a malware probability "
            f"of {probability*100:.1f}%, below the configured "
            f"{threshold*100:.0f}% decision threshold."
        )

        if not signed:
            explanation += (
                " The file is not digitally signed, so the benign prediction "
                "should not be interpreted as a guarantee of safety."
            )

        recommendation = (
            "No malware escalation is required based on this analysis."
        )

    output_text = (
        f"Risk assessment: {risk}\n\n"
        f"{explanation}\n\n"
        f"Recommended action: {recommendation}"
    )

    return {
        "instruction": instruction,
        "input": input_text,
        "output": output_text
    }

In [4]:
examples = []

THRESHOLD = 0.10

file_sizes = ["Small", "Medium", "Large"]

for _ in range(80):
    probability = random.uniform(0.70, 0.999)

    examples.append(
        build_example(
            classification="malware",
            probability=probability,
            threshold=THRESHOLD,
            file_size_group=random.choice(file_sizes),
            max_entropy=random.uniform(5.5, 8.0),
            signed=random.choice([True, False]),
            confidence_band="high"
        )
    )

for _ in range(50):
    probability = random.uniform(0.10, 0.25)

    examples.append(
        build_example(
            classification="malware",
            probability=probability,
            threshold=THRESHOLD,
            file_size_group=random.choice(file_sizes),
            max_entropy=random.uniform(4.5, 7.5),
            signed=random.choice([True, False]),
            confidence_band="low"
        )
    )

for _ in range(70):
    probability = random.uniform(0.001, 0.099)

    examples.append(
        build_example(
            classification="benign",
            probability=probability,
            threshold=THRESHOLD,
            file_size_group=random.choice(file_sizes),
            max_entropy=random.uniform(3.5, 7.0),
            signed=random.choice([True, False]),
            confidence_band="low"
        )
    )

random.shuffle(examples)

print("Total examples:", len(examples))

Total examples: 200


In [5]:
for example in examples[:5]:
    print("=" * 80)
    print("INSTRUCTION:")
    print(example["instruction"])

    print("\nINPUT:")
    print(example["input"])

    print("\nOUTPUT:")
    print(example["output"])
    print()

INSTRUCTION:
Generate a concise malware triage report from the supplied analysis evidence.

INPUT:
Classification: benign
Malware probability: 0.016
Decision threshold: 0.10
File size group: Medium
Maximum entropy: 5.08
Digitally signed: no

OUTPUT:
Risk assessment: Low

The executable was classified as benign with a malware probability of 1.6%, below the configured 10% decision threshold. The file is not digitally signed, so the benign prediction should not be interpreted as a guarantee of safety.

Recommended action: No malware escalation is required based on this analysis.

INSTRUCTION:
Generate a concise malware triage report from the supplied analysis evidence.

INPUT:
Classification: malware
Malware probability: 0.746
Decision threshold: 0.10
File size group: Medium
Maximum entropy: 6.04
Digitally signed: yes

OUTPUT:
Risk assessment: High

The executable was classified as malicious with a probability of 74.6%, which is well above the configured 10% decision threshold.

Recommend

In [6]:
required_keys = {"instruction", "input", "output"}

invalid_examples = []

for i, example in enumerate(examples):
    if set(example.keys()) != required_keys:
        invalid_examples.append(i)

print("Invalid examples:", len(invalid_examples))

Invalid examples: 0


In [7]:
# dataframe for inspection
df_examples = pd.DataFrame(examples)

display(df_examples.head())

,instruction,input,output
0,Generate a concise malware triage report from ...,Classification: benign\nMalware probability: 0...,Risk assessment: Low\n\nThe executable was cla...
1,Generate a concise malware triage report from ...,Classification: malware\nMalware probability: ...,Risk assessment: High\n\nThe executable was cl...
2,Generate a concise malware triage report from ...,Classification: benign\nMalware probability: 0...,Risk assessment: Low\n\nThe executable was cla...
3,Generate a concise malware triage report from ...,Classification: benign\nMalware probability: 0...,Risk assessment: Low\n\nThe executable was cla...
4,Generate a concise malware triage report from ...,Classification: malware\nMalware probability: ...,Risk assessment: High\n\nThe executable was cl...


In [8]:
edge_examples = []

In [9]:
train_examples, temp_examples = train_test_split(
    examples,
    test_size=0.20,
    random_state=42
)

val_examples, test_examples = train_test_split(
    temp_examples,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_examples))
print("Validation:", len(val_examples))
print("Test:", len(test_examples))

Train: 160
Validation: 20
Test: 20


In [10]:
def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [11]:
save_jsonl(
    train_examples,
    LLM_DATA_DIR / "train.jsonl"
)

save_jsonl(
    val_examples,
    LLM_DATA_DIR / "validation.jsonl"
)

save_jsonl(
    test_examples,
    LLM_DATA_DIR / "test.jsonl"
)

save_jsonl(
    examples,
    LLM_DATA_DIR / "shredr_instructions.jsonl"
)

print("Dataset saved.")

Dataset saved.


In [12]:
for file in LLM_DATA_DIR.glob("*.jsonl"):
    print(file.name, file.stat().st_size, "bytes")

shredr_instructions.jsonl 109437 bytes
test.jsonl 10815 bytes
train.jsonl 87129 bytes
validation.jsonl 11493 bytes


In [13]:
edge_examples = []

In [14]:
for probability in [0.82, 0.91, 0.97, 0.995]:
    edge_examples.append({
        "instruction":
            "Generate a concise malware triage report from the supplied analysis evidence.",

        "input":
            f"Classification: malware\n"
            f"Malware probability: {probability:.3f}\n"
            f"Decision threshold: 0.10\n"
            f"File size group: Medium\n"
            f"Maximum entropy: 5.40\n"
            f"Digitally signed: yes",

        "output":
            f"Risk assessment: High\n\n"
            f"The executable was classified as malicious with a probability "
            f"of {probability*100:.1f}%, substantially above the configured "
            f"10% decision threshold. Although the executable is digitally "
            f"signed, signing status does not override the classifier result.\n\n"
            f"Recommended action: Do not execute the file. Escalate it for analyst review."
    })

In [15]:
for probability in [0.012, 0.035, 0.061, 0.089]:
    edge_examples.append({
        "instruction":
            "Generate a concise malware triage report from the supplied analysis evidence.",

        "input":
            f"Classification: benign\n"
            f"Malware probability: {probability:.3f}\n"
            f"Decision threshold: 0.10\n"
            f"File size group: Large\n"
            f"Maximum entropy: 7.75\n"
            f"Digitally signed: yes",

        "output":
            f"Risk assessment: Low\n\n"
            f"The executable was classified as benign with a malware probability "
            f"of {probability*100:.1f}%, below the configured 10% decision threshold. "
            f"The high entropy value is an indicator requiring context and does not "
            f"by itself establish malicious behaviour.\n\n"
            f"Recommended action: No malware escalation is required based on this analysis."
    })

In [16]:
for probability in [0.101, 0.105, 0.110, 0.120, 0.135]:
    edge_examples.append({
        "instruction":
            "Generate a concise malware triage report from the supplied analysis evidence.",

        "input":
            f"Classification: malware\n"
            f"Malware probability: {probability:.3f}\n"
            f"Decision threshold: 0.10\n"
            f"File size group: Small\n"
            f"Maximum entropy: 5.80\n"
            f"Digitally signed: no",

        "output":
            f"Risk assessment: Review required\n\n"
            f"The executable was classified as malicious with a probability "
            f"of {probability*100:.1f}%. The score is only slightly above the "
            f"configured 10% decision threshold, so the result should be treated "
            f"as a low-confidence escalation rather than confirmation of malware.\n\n"
            f"Recommended action: Hold the file for human review before any further action."
    })

In [17]:
for probability in [0.099, 0.095, 0.090, 0.085]:
    edge_examples.append({
        "instruction":
            "Generate a concise malware triage report from the supplied analysis evidence.",

        "input":
            f"Classification: benign\n"
            f"Malware probability: {probability:.3f}\n"
            f"Decision threshold: 0.10\n"
            f"File size group: Medium\n"
            f"Maximum entropy: 6.90\n"
            f"Digitally signed: no",

        "output":
            f"Risk assessment: Review recommended\n\n"
            f"The executable was classified as benign with a malware probability "
            f"of {probability*100:.1f}%, narrowly below the configured 10% threshold. "
            f"Because the prediction is close to the decision boundary, the benign "
            f"classification should be interpreted cautiously.\n\n"
            f"Recommended action: Consider human review if additional security indicators are present."
    })

In [18]:
examples.extend(edge_examples)

random.shuffle(examples)

print("Original examples: 200")
print("Edge cases added:", len(edge_examples))
print("New total:", len(examples))

Original examples: 200
Edge cases added: 17
New total: 217


In [19]:
example_strings = [
    json.dumps(example, sort_keys=True)
    for example in examples
]

duplicate_count = (
    len(example_strings)
    - len(set(example_strings))
)

print("Exact duplicate examples:", duplicate_count)

Exact duplicate examples: 0


In [20]:
invalid_examples = []

for i, example in enumerate(examples):
    if set(example.keys()) != {"instruction", "input", "output"}:
        invalid_examples.append(i)

print("Invalid examples:", len(invalid_examples))

Invalid examples: 0


In [21]:
dataset_manifest = {
    "dataset_name": "SHREDR AI Instruction Dataset",
    "purpose": "Fine-tuning a language model to generate malware triage reports",
    "total_examples": len(examples),
    "construction": (
        "Synthetic instruction-response examples constructed specifically "
        "for the SHREDR AI assessment prototype."
    ),
    "input_fields": [
        "classification",
        "malware_probability",
        "decision_threshold",
        "file_size_group",
        "maximum_entropy",
        "digitally_signed"
    ],
    "target_output": [
        "risk_assessment",
        "evidence_interpretation",
        "recommended_action"
    ],
    "split": {
        "training": len(train_examples),
        "validation": len(val_examples),
        "test": len(test_examples)
    },
    "random_seed": 42,
    "external_personal_data": False,
    "malware_binaries_included": False,
    "licensing_note": (
        "No third-party text corpus is redistributed in this dataset. "
        "Examples were constructed for this project."
    ),
    "ai_assistance_note": (
        "Generative AI assisted with dataset template and example construction; "
        "this use must be disclosed/cited according to course requirements."
    )
}

with open(
    LLM_DATA_DIR / "dataset_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(dataset_manifest, f, indent=2)

print("Dataset manifest saved.")

Dataset manifest saved.
